# 05 – Analysunderlag för ARM

**Syfte:** Slå ihop de bearbetade tabellerna och göra om dem till **transaktioner** som associationsregler kan köras på. En transaktion är en kommun ett visst år. Varje transaktion innehåller ett antal *items*, till exempel `kommungrupp=C`, `bnp_indikator=svag` och `fruktsamhet=hög`.

| | |
|---|---|
| **Indata** | `data/processed/fertilitet_kommun.csv` (02), `befolkningstathet.csv` (03), `konjunktur_ar.csv` (04) |
| **Resultat** | `data/processed/arm_underlag_lag1.csv`, `data/processed/arm_underlag_lag2.csv` |
| **Beroenden** | Kör `01`–`04` först. |

### Frågeställning

> *I vilka kommuntyper ligger fruktsamheten högt eller lågt jämfört med andra kommuner, när konjunkturen året innan var svag eller stark?*

Frågan är inte *"sänker lågkonjunktur fruktsamheten?"*. Konjunkturen är **nationell**, så alla kommuner har samma konjunkturläge ett givet år. Det ger bara 26 oberoende observationer, och ARM är fel verktyg för så få. Utfallet `skillnad_mot_riket` har dessutom redan fått den nationella trenden borttagen. Det datan *kan* besvara är om kommuntyperna **skiljer sig åt** när konjunkturen förändras.

### Beslut som genomförs
| Beslut | Val |
|---|---|
| Utfall | `skillnad_mot_riket` i tertiler: *låg / medel / hög* |
| Konjunkturindikatorer | BNP-indikator, hushållens konsumtion, sysselsättning |
| Konjunkturklasser | *svag* (≤ −0,5) / *normal* / *stark* (> 0,5) |
| Kommuntyp | SKR:s A/B/C |
| Fördröjning (lag) | 1 år i huvudanalysen, 2 år som känslighetsanalys |
| Träning / test | Klassgränser tas fram på 2003–2015 och testas på 2016–2025 |

## 1. Inställningar och parametrar

Alla val från tabellen ovan samlas här, så att de går att ändra på ett ställe.

- `FORSTA_AR = 2003`: första utfallsåret. Med 2 års fördröjning behöver 2003 konjunkturen från 2001. *Sysselsättning* saknas för 2000 (serien startar 2001). Med 2003 som start blir perioden **samma för båda fördröjningarna**, så att resultaten går att jämföra.
- `TRANING` / `TEST`: den tidiga perioden används för att ta fram klassgränser och regler. Den senare perioden används bara för att pröva dem.

In [1]:
from pathlib import Path

import pandas as pd

# Notebooken körs normalt från mappen Assignment.
# Körs den från projektroten letar vi upp Assignment ändå.
BAS = Path.cwd()
if not (BAS / "data" / "raw").is_dir() and (BAS / "Assignment").is_dir():
    BAS = BAS / "Assignment"

RAW = BAS / "data" / "raw"
PROCESSED = BAS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), "Hittar inte data/raw. Öppna notebooken från mappen Assignment."

INDIKATORER = ["bnp_indikator", "hushallens_konsumtion", "sysselsattning"]
LAGAR = [1, 2]
KONJ_GRANS = 0.5

FORSTA_AR = 2003
SISTA_AR = 2025
TRANING = (2003, 2015)
TEST = (2016, 2025)

## 2. Läs in de bearbetade tabellerna

`Kommunkod` läses som text, så att inledande nollor behålls. Vi kontrollerar att nycklarna är unika. Varje sammankoppling i nästa steg kan då kontrolleras med `validate`.

In [2]:
fert = pd.read_csv(PROCESSED / "fertilitet_kommun.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})
tathet = pd.read_csv(PROCESSED / "befolkningstathet.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})
konj = pd.read_csv(PROCESSED / "konjunktur_ar.csv", encoding="utf-8-sig")

assert not fert.duplicated(["Kommunkod", "ar"]).any()
assert not tathet.duplicated(["Kommunkod", "ar"]).any()
assert not konj.duplicated("ar").any()
assert set(INDIKATORER) <= set(konj.columns)

print("Fertilitet:", fert.shape, "| Täthet:", tathet.shape, "| Konjunktur:", konj.shape)

Fertilitet: (7384, 9) | Täthet: (7384, 9) | Konjunktur: (26, 14)


## 3. Koppla ihop fertilitet och kommuntyp

Fertilitet och täthet har samma kommuner och år (det kontrollerades i `03`), så sammankopplingen ska vara **ett-till-ett**. Från täthetstabellen behövs bara kommungruppen, eftersom det är den som ingår som item. Vi kontrollerar också att kommunnamnen stämmer överens på varje rad.

In [3]:
bas = fert.merge(
    tathet[["Kommunkod", "ar", "Kommun", "kommungrupp"]],
    on=["Kommunkod", "ar"], how="inner", validate="one_to_one", suffixes=("", "_tathet"),
)
assert len(bas) == len(fert)
assert (bas["Kommun"] == bas["Kommun_tathet"]).all()
bas = bas.drop(columns="Kommun_tathet")

print("Rader efter sammankoppling:", len(bas))

Rader efter sammankoppling: 7384


## 4. Avgränsa perioden och ta bort rader utan utfall

Två filter, och antalet rader som försvinner i varje redovisas:

1. **Perioden 2003–2025** (se steg 1).
2. **Rader där fruktsamheten saknas.** En transaktion utan utfall kan inte bidra till en regel om fruktsamhet. Bortfallet är redan hanterat och flaggat i `02`. Här tas bara de rader bort där värdet fortfarande saknas.

In [4]:
fore = len(bas)
bas = bas.loc[bas["ar"].between(FORSTA_AR, SISTA_AR)]
efter_period = len(bas)
bas = bas.loc[bas["skillnad_mot_riket"].notna()].reset_index(drop=True)

print(f"Utanför perioden {FORSTA_AR}–{SISTA_AR}: {fore - efter_period} rader")
print(f"Saknad fruktsamhet inom perioden:  {efter_period - len(bas)} rader")
print(f"Kvar: {len(bas)} kommun-år i {bas['Kommunkod'].nunique()} kommuner")

Utanför perioden 2003–2025: 852 rader
Saknad fruktsamhet inom perioden:  54 rader
Kvar: 6478 kommun-år i 284 kommuner


## 5. Dela upp i träning och test

Uppdelningen görs **i tid**, inte slumpmässigt. En slumpmässig uppdelning skulle lägga kommuner från samma år i både träning och test. Konjunkturen är densamma för alla kommuner ett givet år, så testet skulle då bestå av år som modellen redan har sett.

Med en tidsuppdelning tas klassgränserna och reglerna fram på 2003–2015 och prövas på år som modellen aldrig har sett.

In [5]:
bas["del"] = "test"
bas.loc[bas["ar"].between(*TRANING), "del"] = "traning"
assert bas.loc[bas["del"] == "test", "ar"].between(*TEST).all()

print(bas.groupby("del").agg(kommun_ar=("ar", "size"), forsta_ar=("ar", "min"), sista_ar=("ar", "max")))

         kommun_ar  forsta_ar  sista_ar
del                                    
test          2808       2016      2025
traning       3670       2003      2015


## 6. Klassindela fruktsamheten i tertiler

`skillnad_mot_riket` delas i tre lika stora klasser: *låg*, *medel* och *hög*. Lika stora klasser gör att varje klass har tillräckligt stöd (support) för att ge regler.

**Gränserna räknas bara på träningsåren** och används sedan oförändrade på teståren. Räknades de på alla år skulle information från testperioden läcka in i modellen.

**Om namnen:** klasserna är relativa till **andra kommuner**, inte till riket. Medianen för `skillnad_mot_riket` är ungefär +0,12 (se `02`), så även klassen *låg* innehåller kommuner som ligger något över rikstalet. *Hög* betyder alltså "bland den tredjedel av kommunerna som ligger högst", inte "över riket". Därför heter klasserna inte *under / nära / över riket*, som i det första förslaget.

In [6]:
FRUKT_KLASSER = ["låg", "medel", "hög"]

gransvarden = bas.loc[bas["del"] == "traning", "skillnad_mot_riket"].quantile([1 / 3, 2 / 3]).tolist()
bas["fruktsamhet_klass"] = pd.cut(
    bas["skillnad_mot_riket"],
    bins=[-float("inf"), *gransvarden, float("inf")],
    labels=FRUKT_KLASSER,
).astype(str)

print(f"Gränser (från träningsåren): låg ≤ {gransvarden[0]:+.3f} < medel ≤ {gransvarden[1]:+.3f} < hög")
display(pd.crosstab(bas["del"], bas["fruktsamhet_klass"], normalize="index")[FRUKT_KLASSER].round(3))

Gränser (från träningsåren): låg ≤ +0.030 < medel ≤ +0.210 < hög


fruktsamhet_klass,låg,medel,hög
del,,,
test,0.287,0.350,0.363
traning,0.338,0.331,0.332


## 7. Koppla på konjunkturen med fördröjning

Fruktsamhet ett år kopplas till konjunkturen **ett (eller två) år tidigare**. Ett barn som föds år *t* har blivit till ungefär nio månader tidigare, och beslutet att skaffa barn fattades ännu tidigare. Konjunkturläget samma år som födseln kan därför knappast ha påverkat det.

Tekniskt flyttas konjunkturtabellens år framåt med `lag` år (`ar + lag`), och sedan kopplas tabellerna ihop på år. Sammankopplingen är **många-till-en**: många kommuner får samma konjunkturvärde samma år.

Båda fördröjningarna byggs här. Den ena ger huvudanalysen och den andra känslighetsanalysen.

In [7]:
underlag = {}
for lag in LAGAR:
    forskjuten = konj[["ar", *INDIKATORER]].assign(ar=konj["ar"] + lag)
    df = bas.merge(forskjuten, on="ar", how="left", validate="many_to_one")
    assert df[INDIKATORER].notna().all().all(), f"Konjunktur saknas för något år med lag {lag}."
    underlag[lag] = df
    print(f"Lag {lag}: fruktsamhet {df['ar'].min()}–{df['ar'].max()} kopplad till konjunktur "
          f"{df['ar'].min() - lag}–{df['ar'].max() - lag}, {len(df)} rader")

Lag 1: fruktsamhet 2003–2025 kopplad till konjunktur 2002–2024, 6478 rader
Lag 2: fruktsamhet 2003–2025 kopplad till konjunktur 2001–2023, 6478 rader


## 8. Klassindela konjunkturen

Konjunkturläget ligger redan på en standardiserad skala där 0 är normalläge (se `04`). Därför används **fasta gränser** som inte behöver tas fram ur data:

- *svag*: ≤ −0,5
- *normal*: mellan −0,5 och +0,5
- *stark*: > +0,5

Tabellerna visar **hur många olika år** som hamnar i varje klass. Det är det viktigaste måttet på hur mycket data som står bakom en regel. En konjunkturklass som bara förekommer två år ger bara två oberoende observationer, oavsett att den gäller tusentals kommun-år. Det tas upp i `06`.

In [8]:
KONJ_KLASSER = ["svag", "normal", "stark"]

for lag, df in underlag.items():
    for ind in INDIKATORER:
        df[f"{ind}_klass"] = pd.cut(
            df[ind], bins=[-float("inf"), -KONJ_GRANS, KONJ_GRANS, float("inf")], labels=KONJ_KLASSER,
        ).astype(str)

    antal_ar = pd.DataFrame({
        ind: df.drop_duplicates("ar").groupby(["del", f"{ind}_klass"]).size() for ind in INDIKATORER
    }).fillna(0).astype(int)
    print(f"\nLag {lag}: antal år per konjunkturklass")
    display(antal_ar)


Lag 1: antal år per konjunkturklass


bnp_indikator  hushallens_konsumtion  sysselsattning
del                                                                 
test    normal              6                      2               3
        stark               3                      6               5
        svag                1                      2               2
traning normal              5                      4               5
        stark               3                      2               3
        svag                5                      7               5


Lag 2: antal år per konjunkturklass


bnp_indikator  hushallens_konsumtion  sysselsattning
del                                                                 
test    normal              6                      2               3
        stark               3                      6               5
        svag                1                      2               2
traning normal              6                      5               4
        stark               3                      2               4
        svag                4                      6               5

## 9. Slutkontroll och spara

För varje fördröjning kontrollerar vi att:

- nyckeln `Kommunkod` + `ar` är unik
- inga items saknas
- alla klasser har ett av de tillåtna värdena.

Filerna innehåller både **klasserna** (det ARM använder) och de **numeriska värdena** de bygger på. Då går det att kontrollera varje klassning i efterhand. Flaggorna från `02` (`interpolerad`, `har_bortfall`) följer med för känslighetsanalysen i `06`.

In [9]:
KOLUMNER = ["Kommunkod", "Kommun", "ar", "del", "kommungrupp",
            "skillnad_mot_riket", "fruktsamhet_klass",
            *INDIKATORER, *[f"{ind}_klass" for ind in INDIKATORER],
            "interpolerad", "har_bortfall"]

for lag, df in underlag.items():
    df = df[KOLUMNER].sort_values(["Kommunkod", "ar"]).reset_index(drop=True)
    klass_kolumner = ["kommungrupp", "fruktsamhet_klass", *[f"{ind}_klass" for ind in INDIKATORER]]

    assert not df.duplicated(["Kommunkod", "ar"]).any()
    assert df[klass_kolumner].notna().all().all()
    assert set(df["fruktsamhet_klass"]) <= set(FRUKT_KLASSER)
    assert set(df["kommungrupp"]) <= {"A", "B", "C"}
    for ind in INDIKATORER:
        assert set(df[f"{ind}_klass"]) <= set(KONJ_KLASSER)

    utfil = PROCESSED / f"arm_underlag_lag{lag}.csv"
    df.to_csv(utfil, index=False, encoding="utf-8-sig")
    print(f"Sparade {len(df)} rader till {utfil.relative_to(BAS)}")

Sparade 6478 rader till data/processed/arm_underlag_lag1.csv
Sparade 6478 rader till data/processed/arm_underlag_lag2.csv
